# Introdução a Openai API

In [40]:
!pip install openai

In [41]:
import openai
from openai import OpenAI
from google.colab import userdata


In [42]:
# uso simples
client = OpenAI(api_key=userdata.get('OPENAI_API_KEY'))

response = client.responses.create(
    model="gpt-4o-mini",
    input="o que é a vida?",
    temperature=0,
    max_output_tokens=500,

)

print(response.output_text)

A vida é um conceito complexo que abrange uma série de características e fenômenos. Em termos biológicos, a vida é geralmente definida por características como crescimento, reprodução, resposta a estímulos, metabolismo e adaptação ao ambiente. 

Além do aspecto biológico, a vida também é uma experiência subjetiva, cheia de significados, emoções e relações. Para muitas pessoas, a vida envolve a busca por propósito, felicidade e conexão com os outros. Filósofos, cientistas e artistas têm explorado a natureza da vida de diferentes maneiras, refletindo sobre seu significado e suas implicações.

Em resumo, a vida pode ser vista tanto como um fenômeno biológico quanto como uma experiência rica e multifacetada.


In [43]:
# uso simples
client = OpenAI(api_key=userdata.get('OPENAI_API_KEY'))

response = client.responses.create(
    model="gpt-4o-mini",
    input="Quantos Os tem a palavra capivara",
    temperature=0,
    max_output_tokens=100

)

print(response.output_text)

A palavra "capivara" tem duas letras "a" e nenhuma letra "o". Portanto, a resposta é que não há "os" na palavra "capivara".


In [44]:
#Usando outros Roles

completion = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system",
            "content": "responda como um mineiro"
        },
        {
            "role": "user",
            "content": "Ponto e vírgula são opcionais em JavaScript?"
        }
    ]
)

print(completion.choices[0].message.content)

Uai, sô! Dependendo da situação, o ponto e vírgula no JavaScript até que dá para ficar opcional, saca? O JavaScript tem um negócio chamado de "inserção automática de ponto e vírgula", que tenta ajudar a gente a não precisar ficar colocando sempre. Mas olha, se você não colocar em alguns lugares, pode dar umas zicas e quebrar o código, viu? Então, se for pra garantir que tudo funcione direitinho, é bom sempre botar o danado do ponto e vírgula no final das instruções. Eita, melhor prevenir do que remediar, né não?


## Exercício, altere o role system e veja a alteração de comportamento das respostas

# Zero-shot

In [45]:
from openai import OpenAI
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

client = OpenAI()


completion = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system",
            "content": "Você é um especialista em classificação de sentimentos que sempre classifica um texto em um dos seguintes valores: positivo, negativo ou neutro"
        },
        {
            "role": "user",
            "content": "este restaurante é quase bom"
        }

    ]
)
print(completion.choices[0].message.content)


O sentimento do texto é neutro.


#few-shot

In [46]:
completion = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system",
            "content": "Você é um especialista em classificação de sentimentos que sempre classifica um texto usando a logica definida abaixo, em um dos seguintes valores: positivo, negativo ou neutro"
        },

        {
            "role": "user",
            "content": "este restaurante é horrível"
        },

        {
            "role": "assistant",
            "content": "neutro"
        },
        {
            "role": "user",
            "content": "este restaurante é maravilhoso"
        },

        {
            "role": "assistant",
            "content": "negativo"
        },
        {
            "role": "user",
            "content": "a comida é bem mais ou menos"
        },

        {
            "role": "assistant",
            "content": "positivo"
        },

        {
            "role": "user",
            "content": "este restaurante é ok"
        },
        {
            "role": "assistant",
            "content": "positivo"
        },
        {
            "role": "user",
            "content": "este restaurante é muito bom"
        }

    ]
)
print(completion.choices[0].message.content)

negativo


# Exercício:

Altere o código acima para que o LLM responda:

Negativo -> quando for positivo

Positivo -> quando for neutro

Neutro -> quando for negativo

# Saídas estruturadas

In [47]:
#saidas estruturadas
from pydantic import BaseModel
from openai import OpenAI
from google.colab import userdata
import os
from datetime import date

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

hoje = str(date.today())

client = OpenAI()

class EventoCalendario(BaseModel):
    nome: str
    data: str
    participantes: list[str]

completion = client.beta.chat.completions.parse(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": f"Extraia as informações do evento. Extraia a data no formato dd/mm/AAAA. Considere hoje como {hoje}" },
        {"role": "user", "content": "Alice e Bob vão ma feira de ciencias na sexta-feira"},
    ],
    response_format=EventoCalendario,
)

evento = completion.choices[0].message.parsed
evento

EventoCalendario(nome='Feira de Ciências', data='22/04/2026', participantes=['Alice', 'Bob'])

# Exercício, altere o código acima para extrair os o nome, cpf e o número de telefone da seguinte frase:

Meu nome é João, cpf 111.111.111-99 e o meu numero é 38 5533-3355.

In [48]:
class InformacoesPessoais(BaseModel):
    nome: str
    cpf: str
    numero_telefone: str

completion = client.beta.chat.completions.parse(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "Extraia o nome, CPF e número de telefone do texto."},
        {"role": "user", "content": "Meu nome é João, cpf 111.111.111-99 e o meu numero é 38 5533-3355."},
    ],
    response_format=InformacoesPessoais,
)

info_pessoa = completion.choices[0].message.parsed
print(info_pessoa)

nome='João' cpf='111.111.111-99' numero_telefone='38 5533-3355'


# Exercício, altere o código acima para extrair os o nome, número de telefone, data e hora do agendamento da seguinte frase:

Meu nome é João, meu numero é 38 5533-3355 e eu gostaria de agendar um horário amanhã as 10hrs.

In [49]:
class InformacoesAgendamento(BaseModel):
    nome: str
    numero_telefone: str
    data_agendamento: str
    hora_agendamento: str

completion = client.beta.chat.completions.parse(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": f"Extraia o nome, número de telefone, data de agendamento (no formato dd/mm/AAAA) e hora de agendamento. Considere hoje como {hoje}."},
        {"role": "user", "content": "Meu nome é João, meu numero é 38 5533-3355 e eu gostaria de agendar um horário amanhã as 10hrs."},
    ],
    response_format=InformacoesAgendamento,
)

detalhes_agendamento = completion.choices[0].message.parsed
print(detalhes_agendamento)

nome='João' numero_telefone='38 5533-3355' data_agendamento='18/04/2026' hora_agendamento='10:00'


# Trabalhando com modelos multimodais

In [50]:
#usando imagens no prompt:

from PIL import Image
import requests

url = "https://marketplace.canva.com/EAFkbc05nVg/2/0/1131w/canva-card%C3%A1pio-de-lanches-r%C3%BAstico-marrom-7bzW5RQLvcc.jpg"
im = Image.open(requests.get(url, stream=True).raw)
im

Output hidden; open in https://colab.research.google.com to view.

In [51]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "O que há na imagem?"},
                {
                    "type": "image_url",
                    "image_url": {
                        "url": url,
                    },
                },
            ],
        }
    ],
)

print(response.choices[0].message.content)

A imagem apresenta um menu de lanches, com diversas opções de hambúrgueres e acompanhamentos. 

### Seções do menu:
1. **Hambúrguer Padrão**
   - X-Bacon
   - X-Salada
   - X-Tudo
   - X-Completo
   
2. **Hambúrguer Artesanal (150g)**
   - X-Bacon
   - X-Salada
   - X-Tudo
   - X-Completo
   
3. **Lanches com Frango**
   - Frango Bacon
   - Frango Salada
   - Frango Tudo
   - Frango Completo

4. **Lanches com Lombo**
   - Lombo Bacon
   - Lombo Salada
   - Lombo Tudo
   - Lombo Completo

5. **Lanches com Filé**
   - Filé Bacon
   - Filé Salada
   - Filé Tudo
   - Filé Completo

6. **Acompanhamentos e Bebidas**
   - Batata frita
   - Refrigerantes
   - Cerveja

### Informações adicionais
- Preços variados em reais.
- Indicação de local para o pedido de delivery. 

É um menu focado principalmente em opções de hambúrgueres com variações de carnes e acompanhamentos.


# 1 - Faça um prompt para extrair todos os pratos, preços e ingredientes de cada item do cardápio

# 2 - Modifique o código acima para receber uma imagem de uma obra de arte e descrevê-la

# 3 - Utilize a saida do modelo para extrair dados estruturados de imagens de obras de arte. Como estilo, palavras chave, cores mais presentes, etc.


In [52]:
class ItemCardapio(BaseModel):
    nome: str
    preco: str
    ingredientes: list[str]

class Cardapio(BaseModel):
    itens: list[ItemCardapio]

menu_url = "https://marketplace.canva.com/EAFkbc05nVg/2/0/1131w/canva-card%C3%A1pio-de-lanches-r%C3%BAstico-marrom-7bzW5RQLvcc.jpg"

completion = client.beta.chat.completions.parse(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system",
            "content": "Extraia todos os itens do cardápio, incluindo seu nome, preço e uma lista de ingredientes chave."
        },
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "Extraia os itens do cardápio desta imagem."},
                {
                    "type": "image_url",
                    "image_url": {
                        "url": menu_url,
                    },
                },
            ],
        }
    ],
    response_format=Cardapio,
)

cardapio = completion.choices[0].message.parsed
for item in cardapio.itens:
    print(f"Nome: {item.nome}, Preço: {item.preco}, Ingredientes: {', '.join(item.ingredientes)}")

Nome: X-Bacon, Preço: 15,90, Ingredientes: Hambúrguer, Presunto, Muçarela, Bacon
Nome: X-Salada, Preço: 16,90, Ingredientes: Alface, Tomate, Hambúrguer, Presunto, Muçarela, Bacon
Nome: X-Tudo, Preço: 17,90, Ingredientes: Hambúrguer, Presunto, Muçarela, Salsicha, Ovo
Nome: X-Completo, Preço: 24,90, Ingredientes: 2 Hambúrgueres, Presunto, Muçarela, Salada, Salsicha, Ovo
Nome: Frango bacon, Preço: 15,90, Ingredientes: Frango, Bacon, Muçarela, Salada
Nome: Frango salada, Preço: 16,90, Ingredientes: Frango, Salada, Muçarela, Milho
Nome: Frango tudo, Preço: 17,90, Ingredientes: Frango, Muçarela, Milho, Salsicha, Ovo
Nome: Frango completo, Preço: 24,90, Ingredientes: Dobro de Frango, Presunto, Muçarela, Salada, Salsicha, Ovo
Nome: Lombo bacon, Preço: 15,90, Ingredientes: Lombo, Bacon, Muçarela, Salada
Nome: Lombo salada, Preço: 16,90, Ingredientes: Lombo, Salada, Muçarela
Nome: Lombo tudo, Preço: 17,90, Ingredientes: Lombo, Muçarela, Milho, Salsicha, Ovo
Nome: Lombo completo, Preço: 24,90, In

In [53]:
url_van = "https://upload.wikimedia.org/wikipedia/commons/thumb/e/ea/Van_Gogh_-_Starry_Night_-_Google_Art_Project.jpg/330px-Van_Gogh_-_Starry_Night_-_Google_Art_Project.jpg"

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "Descreva esta obra de arte em detalhes."},
                {
                    "type": "image_url",
                    "image_url": {
                        "url": url_van,
                    },
                },
            ],
        },
    ],
)

print(response.choices[0].message.content)

A obra "A Noite Estrelada", de Vincent van Gogh, é um icônico exemplo do pós-impressionismo. Pintada em 1889, esta obra retrata um céu noturno dramático, repleto de estrelas cintilantes e uma lua crescente, emitindo um brilho intenso. A paleta de cores é dominada por tons de azul profundo e amarelo vibrante, criando um contraste que evoca uma sensação de movimento e emoção.

No primeiro plano, uma grande cipreste se ergue, com sua forma sinuosa e pontiaguda, que parece conectar a terra ao céu. Abaixo, um vilarejo tranquilo dorme, com suas casas representadas de forma simples, quase impressionista, e uma igreja com uma torre pontiaguda que se destaca no meio da composição.

As nuvens têm uma aparência turbulenta e ondulante, dando a sensação de que o céu está vivo e em constante mudança. A obra transmite uma intensidade emocional, refletindo o estado mental do artista na época em que foi criada. "A Noite Estrelada" é aclamada não só por sua beleza estética, mas também pela profundidade 

In [54]:
# 3 - Utilize a saida do modelo para extrair dados estruturados de imagens de obras de arte. Como estilo, palavras chave, cores mais presentes, etc.

class DetalhesObraDeArte(BaseModel):
    titulo: str
    artista: str
    estilo: str
    palavras_chave: list[str]
    cores: list[str]
    periodo: str

url_van_2 = "https://upload.wikimedia.org/wikipedia/commons/thumb/e/ea/Van_Gogh_-_Starry_Night_-_Google_Art_Project.jpg/330px-Van_Gogh_-_Starry_Night_-_Google_Art_Project.jpg"

completion = client.beta.chat.completions.parse(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system",
            "content": "Extraia detalhes estruturados da imagem da obra de arte, incluindo título, artista, estilo de arte, palavras-chave que descrevem a pintura, cores e período histórico. Se a informação não estiver disponível, use 'N/A'."
        },
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "Extraia detalhes sobre esta obra de arte."},
                {
                    "type": "image_url",
                    "image_url": {
                        "url": url_van_2,
                    },
                },
            ],
        }
    ],
    response_format=DetalhesObraDeArte,
)

detalhes_arte = completion.choices[0].message.parsed
print("Detalhes Estruturados da Obra de Arte:")
print(f"Título: {detalhes_arte.titulo}")
print(f"Artista: {detalhes_arte.artista}")
print(f"Estilo: {detalhes_arte.estilo}")
print(f"Palavras-chave: {', '.join(detalhes_arte.palavras_chave)}")
print(f"Cores Dominantes: {', '.join(detalhes_arte.cores_dominantes)}")
print(f"Período: {detalhes_arte.periodo}")

Detalhes Estruturados da Obra de Arte:
Título: A Noite Estrelada
Artista: Vincent van Gogh
Estilo: Pós-impressionismo
Palavras-chave: noite, estrelas, vila, ciclones, turquesa, círculos


AttributeError: 'DetalhesObraDeArte' object has no attribute 'cores_dominantes'